# 03 - The testbed

Week 2, Task 3. Build synthetic image pairs where the displacement is known, so
the error of any tracking method can be measured exactly.

Real satellite images never tell you how far the ground moved. Everything
downstream — detection limits, the method comparison, the alert thresholds —
rests on being able to manufacture a known answer.

See `docs/testbed-guide.md` for the reasoning behind each step.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from scipy import ndimage

ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data" / "chamoli"
FIG_DIR = DATA_DIR / "testbed_examples"
FIG_DIR.mkdir(exist_ok=True)

settings = pd.read_csv(DATA_DIR / "settings.csv", header=None, index_col=0).squeeze("columns")
BASE_DATE = settings["base_date"]
print("base image:", BASE_DATE)

raw = np.load(DATA_DIR / f"{BASE_DATE}.npy").astype("float64")
base = (raw - raw.min()) / (raw.max() - raw.min())      # scale to 0-1
BASE_STD = float(base.std())
print(f"shape {base.shape}, range {base.min():.3f}-{base.max():.3f}, std {BASE_STD:.4f}")

## 2. The block that moves

`settings.csv` holds the scar box measured in notebook 02. That box is the
darkest core of the change, not the whole source area: it comes out around
0.04 km2 against the published figure of about 0.25 km2.

It is also too small to track. A cross-correlation window is typically 32 or 64
pixels across, so a patch narrower than that gives the tracker nothing to work
with. The box is expanded here to match the published area, centred on the same
place.

In [ ]:
TARGET_AREA_KM2 = 0.25          # published size of the collapsed block
PIXEL_M = 10

side = int(np.sqrt(TARGET_AREA_KM2 * 1e6) / PIXEL_M)     # pixels
half = side // 2
cr, cc = int(settings["scar_row"]), int(settings["scar_col"])

BOX = (cr - half, cr + half, cc - half, cc + half)       # r0, r1, c0, c1
r0, r1, c0, c1 = BOX

measured = (int(settings["scar_r1"]) - int(settings["scar_r0"])) * \
           (int(settings["scar_c1"]) - int(settings["scar_c0"])) * PIXEL_M**2 / 1e6
print(f"measured scar box: {measured:.3f} km2  (too small to track)")
print(f"using instead:     rows {r0}-{r1}, cols {c0}-{c1} = {side}x{side} px = "
      f"{(side*PIXEL_M)**2/1e6:.3f} km2")

R, C = base.shape
margins = {"north": r0, "south": R - r1, "west": c0, "east": C - c1}
print("\nmargin around the block:",
      "  ".join(f"{k} {v}px" for k, v in margins.items()))
if min(margins.values()) < 32:
    print("WARNING: less than one tracking window of stable ground on one side.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
lo, hi = np.percentile(base, [2, 98])
ax.imshow(np.clip((base - lo) / (hi - lo), 0, 1), cmap="gray")
ax.add_patch(Rectangle((c0, r0), c1 - c0, r1 - r0, fill=False, ec="#ff3b30", lw=2))
ax.set_title(f"{BASE_DATE}: the block the testbed moves")
ax.axis("off")
plt.show()

## 3. Move the block by a known sub-pixel amount

Sub-pixel shifts need interpolation: you cannot move an array 0.2 pixels by
changing indices, so values are estimated between the samples you have.

### Why both images are shifted

Interpolating at a fractional offset smooths an image slightly. Shift only the
"after" image and the two differ in sharpness as well as position, and a tracker
can lock onto that instead of the motion.

An obvious-looking fix is to pass the "before" image through the same function
with a shift of zero. **That does not work.** Cubic spline interpolation is
exact at integer positions, so a zero shift returns the original untouched
(verified: maximum difference about 1e-16).

What does work is splitting the displacement: shift "before" by −d/2 and "after"
by +d/2. The relative displacement is still d, and both images receive the same
amount of interpolation.

In [ ]:
def move_patch(img, dx, dy, box, order=3):
    """Build a before/after pair where `box` has moved by (dx, dy) pixels.

    The displacement is split between the two images so both get equal
    interpolation smoothing. Everything outside `box` is identical.
    """
    r0, r1, c0, c1 = box
    before = ndimage.shift(img, (-dy / 2, -dx / 2), order=order, mode="nearest")
    moved = ndimage.shift(img, (dy / 2, dx / 2), order=order, mode="nearest")
    after = before.copy()
    after[r0:r1, c0:c1] = moved[r0:r1, c0:c1]
    return before, after


# Check: outside the block the two images must be identical.
b, a = move_patch(base, 0.5, 0.0, BOX)
outside = np.ones(base.shape, bool)
outside[r0:r1, c0:c1] = False
print("max difference outside the block:", np.abs(b - a)[outside].max())
print("mean |difference| inside the block:", np.abs(b - a)[~outside].mean().round(5))

In [ ]:
# Can you see it? A few pixels should be obvious; a fifth of a pixel should not.
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, dx in zip(axes, [0.0, 0.2, 3.0]):
    b, a = move_patch(base, dx, 0.0, BOX)
    d = (a - b)[r0:r1, c0:c1]
    ax.imshow(d, cmap="RdBu", vmin=-0.1, vmax=0.1)
    ax.set_title(f"shift {dx} px\nmax |change| {np.abs(d).max():.4f}")
    ax.axis("off")
plt.suptitle("difference inside the block (red/blue = changed)")
plt.tight_layout()
plt.show()

## 4. Noise

The level is set relative to the image's own contrast, so "moderate noise" means
the same thing whichever image or band is used.

### The trap

The two images must get **independent** noise. Add the same pattern to both and
cross-correlation locks onto the noise itself, which matches perfectly, and
reports excellent accuracy that means nothing at all. The failure is silent.

In [ ]:
def add_noise(img, level, rng, base_std=None):
    """Add Gaussian noise. `level` is a fraction of the image's own contrast."""
    sigma = level * (BASE_STD if base_std is None else base_std)
    return img + rng.normal(0, sigma, img.shape)


rng = np.random.default_rng(0)
n1 = add_noise(base, 0.1, rng) - base
n2 = add_noise(base, 0.1, rng) - base
print(f"requested sigma {0.1 * BASE_STD:.5f}  measured {n1.std():.5f}")
print(f"correlation between the two noise draws: {np.corrcoef(n1.ravel(), n2.ravel())[0, 1]:+.5f}")
print("(should be near zero - if it is near 1, both images got the same noise)")

## 5. Clouds

Real cloud leaves holes, not speckle. Scattered single missing pixels are easy to
fill from their neighbours; a 50-pixel hole is not. Modelling cloud as random
pixels makes the problem look far easier than it is.

Method: generate a random field, smooth it heavily so nearby values become
similar, then cut it at the quantile matching the fraction wanted. The quantile
guarantees the exact fraction however much smoothing is applied.

Missing pixels are marked `NaN` rather than given a brightness value, so every
later step has to handle the gap deliberately. The mask is returned because the
restoration step in week 4 needs to know which pixels are real.

In [ ]:
def add_clouds(img, fraction, rng, smooth=15):
    """Blank out `fraction` of pixels as cloud blobs. Returns (image, mask)."""
    if fraction <= 0:
        return img.copy(), np.zeros(img.shape, bool)
    field = ndimage.gaussian_filter(rng.random(img.shape), smooth)
    mask = field > np.quantile(field, 1 - fraction)
    out = img.copy()
    out[mask] = np.nan
    return out, mask


for f in [0.1, 0.3, 0.6]:
    out, mask = add_clouds(base, f, np.random.default_rng(1))
    print(f"requested {f:.2f}  got {mask.mean():.4f}  "
          f"mask matches NaNs: {np.array_equal(mask, np.isnan(out))}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, f in zip(axes, [0.0, 0.15, 0.35, 0.6]):
    out, _ = add_clouds(base, f, np.random.default_rng(1))
    ax.imshow(out, cmap="gray", vmin=0.2, vmax=0.8)
    ax.set_title(f"{int(f * 100)}% cloud")
    ax.axis("off")
plt.suptitle("white gaps are missing data, not bright ground")
plt.tight_layout()
plt.show()

## 6. One function

Order of operations matches reality: the ground moves, the sensor adds noise,
the atmosphere blocks the view.

Everything is driven by one seed. This matters beyond tidiness: when three
methods are compared in week 6 they must see **identical** inputs, otherwise
part of the measured difference is luck rather than method.

In [ ]:
def make_test_pair(dx, dy, noise=0.1, cloud=0.0, seed=0, img=None, box=None):
    """One synthetic pair with a known displacement.

    Returns (before, after, mask, true_shift) where true_shift is (dx, dy).
    """
    img = base if img is None else img
    box = BOX if box is None else box
    rng = np.random.default_rng(seed)

    before, after = move_patch(img, dx, dy, box)
    before = add_noise(before, noise, rng)
    after = add_noise(after, noise, rng)
    after, mask = add_clouds(after, cloud, rng)
    return before, after, mask, (dx, dy)


b1, a1, m1, t1 = make_test_pair(0.5, 0.0, noise=0.1, cloud=0.2, seed=42)
b2, a2, m2, t2 = make_test_pair(0.5, 0.0, noise=0.1, cloud=0.2, seed=42)
b3, a3, m3, t3 = make_test_pair(0.5, 0.0, noise=0.1, cloud=0.2, seed=43)
print("same seed gives identical output:", np.array_equal(a1, a2, equal_nan=True))
print("different seed differs:          ", not np.array_equal(a1, a3, equal_nan=True))
print("true shift returned:", t1)

## 7. Does it round-trip?

Impose a known shift, measure it back, compare. This is the check that says
whether the testbed is trustworthy.

A minimal tracker is used here only for checking. Week 3 builds the real one.

**One thing this check revealed.** Correlating the raw patches underestimates
small shifts badly: a true 0.5 px came back as 0.24, and 1.0 px as 0.76. The
cause is the sharp edge of the analysis window, which leaks across the frequency
spectrum and biases the result. Tapering the patch with a Hann window before
correlating fixes it. That taper belongs in the week 3 tracker.

In [ ]:
from skimage.registration import phase_cross_correlation

INSET = 6          # stay clear of the block's edges


def taper(a):
    """Smoothly fade the patch edges to zero, to stop them biasing the correlation."""
    h = np.hanning(a.shape[0])[:, None] * np.hanning(a.shape[1])[None, :]
    return (a - np.nanmean(a)) * h


def measure(before, after, box=None, use_taper=True):
    r0, r1, c0, c1 = BOX if box is None else box
    w = (slice(r0 + INSET, r1 - INSET), slice(c0 + INSET, c1 - INSET))
    f = taper if use_taper else (lambda x: x)
    est, _, _ = phase_cross_correlation(
        f(before[w]), f(after[w]), upsample_factor=100, normalization=None
    )
    return -est[1], -est[0]          # dx, dy


print(" true dx | raw patch | tapered")
for dx in [0.0, 0.2, 0.5, 1.0, 2.0, 3.0]:
    b, a = move_patch(base, dx, 0.0, BOX)
    raw_dx, _ = measure(b, a, use_taper=False)
    tap_dx, _ = measure(b, a, use_taper=True)
    print(f"   {dx:4.1f}   |  {raw_dx:+6.3f}   | {tap_dx:+6.3f}")

### Zero-shift control

No displacement, only noise. The tracker should report about zero. A consistent
offset in one direction would mean something is biased.

In [ ]:
errs = []
for seed in range(12):
    b, a, _, _ = make_test_pair(0.0, 0.0, noise=0.1, seed=seed)
    dx, dy = measure(b, a)
    errs.append((dx, dy))
errs = np.array(errs)
print(f"mean  dx {errs[:, 0].mean():+.4f}   dy {errs[:, 1].mean():+.4f}   (want ~0)")
print(f"std   dx {errs[:, 0].std():.4f}    dy {errs[:, 1].std():.4f}")
print("\nThe spread is the noise floor: motion smaller than this cannot be")
print("told apart from measurement error. That is the detection limit, measured")
print("properly for the first time in week 6.")

## 8. Save examples for the report

In [ ]:
cases = [("clean", dict(noise=0.02, cloud=0.0)),
         ("noisy", dict(noise=0.35, cloud=0.0)),
         ("cloudy", dict(noise=0.1, cloud=0.35))]

fig, axes = plt.subplots(len(cases), 2, figsize=(10, 5 * len(cases)))
for row, (name, kw) in zip(axes, cases):
    b, a, _, _ = make_test_pair(1.0, 0.0, seed=7, **kw)
    for ax, (img, label) in zip(row, [(b, "before"), (a, "after")]):
        ax.imshow(img[r0 - 20:r1 + 20, c0 - 20:c1 + 20], cmap="gray", vmin=0.2, vmax=0.8)
        ax.set_title(f"{name}: {label}")
        ax.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "testbed_examples.png", dpi=120, bbox_inches="tight")
plt.show()
print("saved", FIG_DIR / "testbed_examples.png")